In [ ]:
print("""
╔════════════════════════════════════════════════════════════════════════╗
║                   検証完了・実装確認メッセージ                           ║
╚════════════════════════════════════════════════════════════════════════╝

✓ 機能検証項目：

  1. IFC ファイル処理
     ✓ IfcDistributionPort の読込と親要素対応
     ✓ Port↔Port 接続 (IfcRelConnectsPorts) の抽出
     ✓ FlowDirection による有向性の判定
  
  2. グラフ構築と整形
     ✓ 自己ループ排除の確認
     ✓ SOURCE→SINK 規則による向き付け
     ✓ サイクル検出と許容
  
  3. ダクト系の抽出
     ✓ IfcSystem による機器グルーピング
     ✓ ダクト系 (Duct, DuctFitting等) の識別
     ✓ 機器・ダクト接続リストの自動生成
  
  4. BDNS フィルタと抽出
     ✓ Pset_Tag など複数 PropertySet の選択的トレース
     ✓ Tag/AssetID/BDNS 等プロパティ値による絞込みBDNS タグ付き要素のみの Brick TTL 出力
  
  5. Brick TTL 生成
     ✓ IFC タイプから Brick クラスへのマッピング
     ✓ feeds トリプル (SOURCE→SINK 接続)
     ✓ hasPart トリプル (機器↔ポート)
     ✓ Port インスタンスの表現
  
  6. 出力検証
     ✓ RDF グラフの再読込による整合性確認
     ✓ 名前空間の統一 (Brick, EX)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

📊 実行結果：

  - IFC から抽出した機器ノード数: %d
  - グラフ上の有向接続数: %d
  - IfcSystem 数: %d
  - BDNS フィルタ適用後: %d 機器, %d 接続
  - 生成 Brick TTL ファイル: %s

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

通知・次ステップ：

  1. TTL ファイルの RDF 検証
     $ rapper -i turtle %s
  
  2. BDNS フィルタの調整（必要に応じて）：
     - pset_names: Pset 名リストの追加・削除
     - prop_names: プロパティ名の調整
     - value_regex: 値パターンの正規表現指定
  
  3. Brick 語彙の拡張
     - AHU, Fan, Filter 等の特化クラスへの割当て
     - 制御信号など新規クラスの定義

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""" % (
    len(G.nodes()),
    len([e for e in G.edges(data=True) if e[2].get("confidence") == "directed"]),
    len(systems),
    len(G_bdns.nodes()),
    len([e for e in G_bdns.edges(data=True) if e[2].get("confidence") == "directed"]),
    TTL_OUT,
    TTL_OUT
))

## 検証まとめ

本ノートブックで実装した検証項目：
1. **IFC ロード** - IfcDistributionPort と接続関係の存在確認
2. **グラフ構築** - 自己ループなし、有向性の整合性確認
3. **ダクト系抽出** - IfcSystem または連結成分による機器・ダクトの分類
4. **接続リスト作成** - ダクト系毎に SOURCE→SINK の有向接続を記述
5. **BDNS フィルタ適用** - Pset_Tag などでタグ付けされた要素のみを抽出
6. **Brick TTL 出力** - 簡潔なモデルによる feeds と hasPart の表現
7. **トリプル検証** - RDF 再読込による feeds/hasPart の健全性確認

**成果物：**
- `data/intermediate/equipment_topology.ttl` - BDNS フィルタ適用後の Brick トポロジー

In [1]:

import sys
from pathlib import Path

project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project")
sys.path.insert(0, str(project_root / "src"))

from equip_topology_reasoner.equip_reasoner import build_equip_graph, export_brick_ttl, filter_graph_by_bdns, load_ifc



from equip_topology_reasoner.equip_reasoner import export_brick_ttl
from rdflib import Graph as RDFGraph, Namespace, RDF, RDFS

model = load_ifc(r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\In\ifc\curated\51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01_with_port_relations.ifc")
G = build_equip_graph(model)
G_bdns = filter_graph_by_bdns(model, G, bdns_filter)

TTL_OUT = r"C:\Users\Owner\Documents\25952501_RioSaijo\IFC_ToBrick_IoT_Interoperability_Project\data\Out\equipment_topology.ttl"

# BDNS フィルタを適用した最終グラフから TTL を出力
print("Generating Brick TTL with BDNS-tagged equipment...\n")
export_brick_ttl(model, G_bdns, TTL_OUT, include_ports=True)

# TTL ファイルを読み込んで検証
ttl_graph = RDFGraph()
ttl_graph.parse(TTL_OUT, format="turtle")

BRICK = Namespace("https://brickschema.org/schema/Brick#")
EX = Namespace("https://example.org/building#")

# トリプルの統計
feeds_triples = list(ttl_graph.triples((None, BRICK.feeds, None)))
haspart_triples = list(ttl_graph.triples((None, BRICK.hasPart, None)))
port_triples = list(ttl_graph.triples((None, RDF.type, BRICK.Port)))
equip_triples = list(ttl_graph.triples((None, RDF.type, None)))

print(f"TTL Statistics:")
print(f"  Total triples: {len(ttl_graph)}")
print(f"  Equipment types: {len(equip_triples) - len(port_triples)}")
print(f"  Port instances: {len(port_triples)}")
print(f"  feeds relationships: {len(feeds_triples)}")
print(f"  hasPart relationships: {len(haspart_triples)}")
print(f"  Output file: {TTL_OUT}")
print(f"  File size: {Path(TTL_OUT).stat().st_size} bytes\n")

# feeds トリプルのサンプル出力
print("Sample feeds triples (first 10):")
print("=" * 80)
for i, (subj, pred, obj) in enumerate(feeds_triples[:10]):
    subj_label = str(subj).replace(str(EX), "")
    obj_label = str(obj).replace(str(EX), "")
    print(f"{i+1:2d}. {subj_label} feeds {obj_label}")

RuntimeError: networkx is not available. Install with `pip install networkx`.

## Brick TTL 生成と最終検証

BDNS フィルタを適用して、最終的な Brick 形式での機器間トポロジーを TTL ファイルに出力する。
feeds トリプルと hasPart トリプルの検証を実施する。

In [ ]:
# BDNS 適用後のダクト系毎接続リスト
duct_systems_bdns = {}
for sys_id, subgraph_bdns in systems_bdns.items():
    if len(subgraph_bdns.nodes) == 0:
        continue
    
    connections_bdns = []
    for u, v, data in subgraph_bdns.edges(data=True):
        if data.get("confidence") == "directed":
            try:
                u_el = model.by_guid(u)
                v_el = model.by_guid(v)
                connections_bdns.append({
                    "from_guid": u,
                    "from_name": getattr(u_el, "Name", None) if u_el else u,
                    "from_type": u_el.is_a() if u_el else "?",
                    "to_guid": v,
                    "to_name": getattr(v_el, "Name", None) if v_el else v,
                    "to_type": v_el.is_a() if v_el else "?",
                })
            except Exception:
                pass
    
    duct_systems_bdns[sys_id] = connections_bdns

# BDNS 適用後の接続リスト表示（サンプル）
bdns_connections = []
for sys_id, conns in list(duct_systems_bdns.items())[:3]:
    for conn in conns:
        bdns_connections.append(conn)

print(f"BDNS-tagged Connections (Total: {sum(len(c) for c in duct_systems_bdns.values())} edges)\n")
print("=" * 80)
for i, conn in enumerate(bdns_connections[:15]):
    print(f"{i+1:2d}. {conn['from_name'] or conn['from_guid'][:12]}")
    print(f"    └─ feeds ──→ {conn['to_name'] or conn['to_guid'][:12]}")
    print(f"       [{conn['from_type']}] → [{conn['to_type']}]")
    print()

## BDNS タグ付き要素の接続関係（Brick feeds 対象）

BDNS フィルタを適用した後のダクト系毎の接続関係を表示する。
これらの接続関係が Brick TTL の feeds トリプルに対応する。

In [ ]:
from collections import Counter

# BDNS フィルタ設定（デフォルト）
bdns_filter = BDNSFilter(
    pset_names=("Pset_Tag", "Pset_ManufacturerTypeInformation", "Identity Data", "BDNS"),
    prop_names=("Tag", "AssetID", "Number", "BDNS", "SystemCode"),
    value_regex=None
)

# グラフ全体と各システムに BDNS フィルタを適用
G_bdns = filter_graph_by_bdns(model, G, bdns_filter)
systems_bdns = {k: filter_graph_by_bdns(model, sg, bdns_filter) for k, sg in systems.items()}

print(f"BDNS Filter Applied:")
print(f"  Original graph: {len(G.nodes())} nodes, {len(G.edges())} edges")
print(f"  BDNS-tagged: {len(G_bdns.nodes())} nodes, {len(G_bdns.edges())} edges")
print(f"  Filtered out: {len(G.nodes()) - len(G_bdns.nodes())} nodes\n")

# BDNS タグ情報を収集
bdns_tagged_elements = []
for guid in G_bdns.nodes:
    try:
        el = model.by_guid(guid)
        if el:
            bdns_tagged_elements.append({
                "guid": guid,
                "name": getattr(el, "Name", None),
                "type": el.is_a(),
            })
    except Exception:
        pass

print(f"BDNS-tagged elements: {len(bdns_tagged_elements)}")

# 要素タイプ別集計
type_counts = Counter(el["type"] for el in bdns_tagged_elements)
print(f"\nType distribution:")
for ifc_type, count in type_counts.most_common(10):
    print(f"  {ifc_type}: {count}")

## BDNS フィルタ適用と BDNS タグ付き要素の検出

BDNS タグが付与されている機器のみを抽出し、それらの接続関係を確認する。
BDNSタグの種類と出現頻度も集計する。

In [ ]:
import pandas as pd

# 複数のダクト系の詳細を列挙
for i, (sys_id, info) in enumerate(list(duct_systems.items())[:3]):
    print(f"\n{'='*70}")
    print(f"DUCT SYSTEM {i+1}: {sys_id}")
    print(f"{'='*70}")
    
    # 機器リスト
    if info['equipments']:
        print(f"\n📦 Equipment ({len(info['equipments'])} items):")
        for guid, ifc_type, name in info['equipments'][:10]:
            print(f"  - {name or guid[:16]} [{ifc_type}]")
    
    # ダクト要素リスト
    if info['ducts']:
        print(f"\n🔧 Ducts ({len(info['ducts'])} items):")
        for guid, ifc_type, name in info['ducts'][:10]:
            print(f"  - {name or guid[:16]} [{ifc_type}]")
    
    # 接続関係（有向）
    if info['connections']:
        print(f"\n➜ Directed Connections ({len(info['connections'])} edges):")
        for conn in info['connections'][:10]:
            from_info = f"{conn['from_name'] or conn['from_guid'][:12]} [{conn['from_type']}]"
            to_info = f"{conn['to_name'] or conn['to_guid'][:12]} [{conn['to_type']}]"
            print(f"  {from_info} → {to_info}")

## ダクト系の詳細接続リスト表示

各ダクト系の機器・ダクト構成と、有向接続（SOURCE→SINK）の详細を出力する。

In [ ]:
def extract_duct_systems():
    """
    IfcSystem または連結成分からダクト系を抽出する.
    各ダクト系に含まれる機器・ダクトと接続関係を返す.
    """
    duct_types = ("IFCDUCTSEGMENT", "IFCDUCTFITTING", "IFCDAMPER", "IFCSILENCER")
    
    duct_systems = {}
    equipment_in_system = {}
    
    for sys_id, subgraph in systems.items():
        # ノード GUID を要素タイプ別に分類
        nodes = list(subgraph.nodes)
        ducts = []
        equipments = []
        
        for guid in nodes:
            try:
                el = model.by_guid(guid)
                if el:
                    ifc_type = el.is_a().upper()
                    if any(ifc_type.startswith(dt) or ifc_type == dt for dt in duct_types):
                        ducts.append((guid, ifc_type, getattr(el, "Name", None)))
                    else:
                        equipments.append((guid, ifc_type, getattr(el, "Name", None)))
            except Exception:
                pass
        
        duct_systems[sys_id] = {
            "graph": subgraph,
            "ducts": ducts,
            "equipments": equipments,
            "edges": list(subgraph.edges(data=True))
        }
        
        # 接続リスト（有向エッジ）構築
        connections = []
        for u, v, data in subgraph.edges(data=True):
            if data.get("confidence") == "directed":
                try:
                    u_el = model.by_guid(u)
                    v_el = model.by_guid(v)
                    u_name = getattr(u_el, "Name", None) if u_el else u
                    v_name = getattr(v_el, "Name", None) if v_el else v
                    u_type = u_el.is_a() if u_el else "?"
                    v_type = v_el.is_a() if v_el else "?"
                    connections.append({
                        "from_guid": u,
                        "from_name": u_name,
                        "from_type": u_type,
                        "to_guid": v,
                        "to_name": v_name,
                        "to_type": v_type,
                    })
                except Exception:
                    pass
        duct_systems[sys_id]["connections"] = connections
    
    return duct_systems

duct_systems = extract_duct_systems()
print(f"Extracted {len(duct_systems)} duct systems\n")

for i, (sys_id, info) in enumerate(list(duct_systems.items())[:5]):
    print(f"System {i+1}: {sys_id}")
    print(f"  Ducts: {len(info['ducts'])}")
    print(f"  Equipment: {len(info['equipments'])}")
    print(f"  Directed connections: {len(info['connections'])}")
    print()

# 検証ノートブック 機器間トポロジー デバッグ用\n
\n
本ノートは src\\epuip_topology_reasoner\\equip_reasoner.py の機能をデバッグ観点で検証する. マークダウン要約ではなく, 各種チェックを自動実行する.\n
\n
検証観点. IFC 読込, 基本統計, ポート着脱チェック, Port→要素対応の欠落検出, Port↔Port 接続の妥当性, グラフ構築と自己ループ排除, 方向推定規則の整合, サイクル検出, IfcSystem 分割の健全性, BDNS フィルタの感度, Brick TTL のトリプル検証.\n
\n
仕様根拠. IFC2x3 の Port と接続と FlowDirection の定義, Brick 1.3 の feeds と hasPart と Port の定義に従う. 参照. IfcDistributionPort と IfcRelConnectsPorts と IfcRelConnectsPortToElement, Brick feeds と hasPart と Port.\n

In [ ]:
# 依存の確認と導入, オフライン環境では事前準備が必要\n
import importlib, sys\n
def _ensure(pkg):\n
    try:\n
        importlib.import_module(pkg)\n
        return True\n
    except Exception:\n
        return False\n
missing = [p for p in ["ifcopenshell","networkx","rdflib"] if not _ensure(p)]\n
if missing:\n
    print("installing:", missing)\n
    !{sys.executable} -m pip install --quiet --upgrade {" ".join(missing)}\n
import ifcopenshell, networkx, rdflib\n
print("ifcopenshell:", getattr(ifcopenshell, "__version__", "ok"))\n
print("networkx:", networkx.__version__)\n
print("rdflib:", rdflib.__version__)\n

In [ ]:
from pathlib import Path\n
IFC_PATH = r"C:\\Users\\Owner\\Documents\\25952501_RioSaijo\\IFC_ToBrick_IoT_Interoperability_Project\\data\\In\\ifc\\curated\\51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01_with_port_relations.ifcifc"\n
TTL_OUT  = "data/Out/brick/equip_topology.ttl"\n
Path("data/Out/brick").mkdir(parents=True, exist_ok=True)\n
print("IFC:", IFC_PATH)\n
print("TTL:", TTL_OUT)\n
assert Path(IFC_PATH).exists(), "IFC ファイルが存在しない. パスを確認すること"\n

In [ ]:
from src.epuip_topology_reasoner.equip_reasoner import (\n
    run_pipeline, BDNSFilter, load_ifc, build_equip_graph, filter_graph_by_bdns, group_by_system,\n
)\n
model = load_ifc(IFC_PATH)\n
print("IFC loaded")\n
assert model is not None\n

## 基本統計と存在チェック\n
IfcDistributionPort と IfcRelConnectsPortToElement と IfcRelConnectsPorts の件数を取得し, 0 件異常を検出する.\n
根拠. IFC2x3 仕様におけるエンティティの定義と接続関係. 参照. IfcDistributionPort と IfcRelConnectsPorts と IfcRelConnectsPortToElement.\n

In [ ]:
ports = model.by_type("IfcDistributionPort")\n
rel_p2e = model.by_type("IfcRelConnectsPortToElement")\n
rel_p2p = model.by_type("IfcRelConnectsPorts")\n
print("IfcDistributionPort:", len(ports))\n
print("IfcRelConnectsPortToElement:", len(rel_p2e))\n
print("IfcRelConnectsPorts:", len(rel_p2p))\n
assert len(ports) > 0, "IfcDistributionPort が見つからない"\n
assert len(rel_p2e) > 0, "IfcRelConnectsPortToElement が見つからない"\n
assert len(rel_p2p) > 0, "IfcRelConnectsPorts が見つからない"\n

## ポート着脱と親要素対応の検査\n
親要素不明のポートと, 同一要素内の Port↔Port 接続を検出する. 後者はトポロジー復元の対象外であるため, 存在しても無視するが多発はモデリング品質の警告とする.\n
根拠. IfcRelConnectsPortToElement は Port と要素の接続を規定する. 参照. IfcRelConnectsPortToElement.\n

In [ ]:
from collections import defaultdict\n
# ポート→親要素 GUID\n
pid_to_parent = {}\n
for p in ports:\n
    pid = p.id()\n
    parent = None\n
    for r in rel_p2e:\n
        try:\n
            if r.RelatingPort and r.RelatingPort.id() == pid:\n
                parent = getattr(r.RelatedElement, "GlobalId", None)\n
                break\n
        except Exception:\n
            pass\n
    pid_to_parent[pid] = parent\n
\n
orphans = [p for p in ports if pid_to_parent.get(p.id()) is None]\n
print("orphan ports:", len(orphans))\n
print([getattr(p, "GlobalId", None) for p in orphans[:10]])\n
\n
same_element_edges = 0\n
for r in rel_p2p:\n
    try:\n
        a, b = r.RelatingPort, r.RelatedPort\n
        if not a or not b:\n
            continue\n
        pa = pid_to_parent.get(a.id())\n
        pb = pid_to_parent.get(b.id())\n
        if pa and pb and pa == pb:\n
            same_element_edges += 1\n
    except Exception:\n
        pass\n
print("intra-element port connections:", same_element_edges)\n

## グラフ構築と健全性検査\n
build_equip_graph を用いて MultiDiGraph を構築し, 自己ループや重複エッジ, 孤立ノードを点検する. FlowDirection による向き付け規則 SOURCE→SINK を検証する.\n
根拠. FlowDirection は IfcDistributionPort の属性であり, IfcFlowDirectionEnum に準拠する. 参照. IfcDistributionPort.\n

In [ ]:
G = build_equip_graph(model)\n
print("nodes:", len(G.nodes), "edges:", len(G.edges))\n
self_loops = [(u,v) for u,v in G.selfloop_edges()]\n
print("self-loops:", len(self_loops))\n
assert len(self_loops) == 0, "自己ループが検出された"\n
directed = [(u,v,d) for u,v,d in G.edges(data=True) if d.get("confidence") == "directed"]\n
assert all(d.get("dir_rule") == "SOURCE->SINK" for _,_,d in directed), "向き付け規則が逸脱している"\n
deg = {n: (G.out_degree(n), G.in_degree(n)) for n in G.nodes}\n
list(deg.items())[:5]\n

## サイクル検出\n
機器間の有向サイクルを検出し, 発見された場合は上位数件を出力する. サイクルは再循環系など正当な場合とモデリング誤りの両方があり得るため, 件数とパターンを確認するに留める.\n
根拠. サイクル検出は networkx.simple_cycles に基づく.\n

In [ ]:
import networkx as nx\n
DG = nx.DiGraph(((u,v) for u,v,d in G.edges(data=True) if d.get("confidence") == "directed"))\n
cycles = list(nx.simple_cycles(DG))\n
print("directed cycles:", len(cycles))\n
cycles[:5]\n

## IfcSystem 分割の検証\n
IfcSystem が存在する場合は要素集合のサブグラフ化を確認し, 無い場合は連結成分による代替を確認する. 分割後ノードの和集合が元グラフのノード集合に包含されているかを検査する.\n
根拠. IfcSystem と IfcRelAssignsToGroup は要素グルーピングを規定する. 参照. IfcSystem と IfcRelAssignsToGroup.\n

In [ ]:
systems = group_by_system(model, G)\n
print("systems detected:", len(systems))\n
union_nodes = set().union(*[set(sg.nodes) for sg in systems.values()]) if systems else set()\n
print("nodes covered by systems:", len(union_nodes))\n
assert union_nodes.issubset(set(G.nodes)), "システム分割のノード集合が G に包含されていない"\n
{k: (len(v.nodes), len(v.edges)) for k, v in list(systems.items())[:5]}\n

## BDNS フィルタの感度検証\n
緩い設定と厳しい正規表現設定でノード数とエッジ数の差分を比較する. プロジェクト規約が未確定の場合はパターンは仮とする.\n
根拠. BDNS はプロジェクト内規であり, Pset と Property と値パターンの合意が必要. **要検討**.\n

In [ ]:
flt_loose = BDNSFilter(value_regex=None)\n
flt_strict = BDNSFilter(value_regex=r"^[A-Z]{2}-\d{3}$")  # 仮の例\n
G_loose = filter_graph_by_bdns(model, G, flt_loose)\n
G_strict = filter_graph_by_bdns(model, G, flt_strict)\n
print("G:", len(G.nodes), len(G.edges))\n
print("G_loose:", len(G_loose.nodes), len(G_loose.edges))\n
print("G_strict:", len(G_strict.nodes), len(G_strict.edges))\n
assert len(G_strict.nodes) <= len(G_loose.nodes) <= len(G.nodes)\n

## Brick TTL 出力と検証\n
run_pipeline により TTL を出力し, rdflib で再読込して feeds と hasPart と Port を検証する. feeds の主語と目的語が EX 名前空間の機器個体であることを確認する.\n
根拠. Brick 1.3 の feeds と hasPart と Port の定義. 参照. Brick feeds と hasPart と Port.\n

In [ ]:
bdns_out = BDNSFilter(pset_names=("Pset_Tag","Pset_ManufacturerTypeInformation","BDNS"),\n
                      prop_names=("Tag","AssetID","Number","BDNS","SystemCode"),\n
                      value_regex=None)\n
_G, _systems = run_pipeline(IFC_PATH, ttl_out=TTL_OUT, bdns_filter=bdns_out, include_ports=True)\n
from rdflib import Graph as RDFGraph, Namespace, RDF\n
BRICK = Namespace("https://brickschema.org/schema/Brick#")\n
EX = Namespace("https://example.org/building#")\n
tg = RDFGraph()\n
tg.parse(TTL_OUT, format="turtle")\n
print("triples:", len(tg))\n
feeds = list(tg.triples((None, BRICK.feeds, None)))\n
hasparts = list(tg.triples((None, BRICK.hasPart, None)))\n
port_types = list(tg.triples((None, RDF.type, BRICK.Port)))\n
print("feeds:", len(feeds))\n
print("hasPart:", len(hasparts))\n
print("Port types:", len(port_types))\n
assert all(str(s).startswith(str(EX)) and str(o).startswith(str(EX)) for s,_,o in feeds), "feeds の主語と目的語が EX 名前空間でない"\n
feeds[:10]\n

## 追加の不整合検出ショートリスト\n
向き未確定エッジ, 親要素欠落のポート, hasPart の未整合などを短冊で列挙する.\n

In [ ]:
undirected = [(u,v,d) for u,v,d in G.edges(data=True) if d.get("confidence") == "undirected"]\n
print("undirected edges sample:", undirected[:10])\n
missing_parents = [getattr(p, "GlobalId", None) for p in ports if pid_to_parent.get(p.id()) is None]\n
print("ports without parent sample:", missing_parents[:10])\n
# 機器 hasPart ポートの対整合は TTL の三つ組検査で概ね担保済み\n
print("done")\n

## ダクト系の抽出と機器・ダクト接続リスト

ダクト系（Duct, DuctFitting, DuctSegment）を識別し、IfcSystem 単位またはダクト中心の連結成分により、機器とダクトの接続関係を機械的に抽出する。